# NARCIS / TOMM — Frozen DiffStega GPU reproduction

This notebook is only a launcher. It does not alter DiffStega, the UniStega corpus, the frozen upstream commit, model identifiers, sampling steps, edit strengths, passwords, or any benchmark command. Enable an NVIDIA GPU accelerator before running.

In [ ]:
import os, pathlib, subprocess, shutil, json
work = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path('/content')
print({'work_root': str(work), 'nvidia_smi': shutil.which('nvidia-smi')})
subprocess.run(['nvidia-smi'], check=True)
repo = work / 'NARCIS'
if repo.exists():
    subprocess.run(['git','-C',str(repo),'fetch','origin','tomm-revision'], check=True)
    subprocess.run(['git','-C',str(repo),'checkout','-f','tomm-revision'], check=True)
    subprocess.run(['git','-C',str(repo),'reset','--hard','origin/tomm-revision'], check=True)
else:
    subprocess.run(['git','clone','--branch','tomm-revision','--single-branch','https://github.com/EkodeckStephane/NARCIS.git',str(repo)], check=True)
print(subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'], text=True).strip())

In [ ]:
import os, pathlib, subprocess
work = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path('/content')
repo = work / 'NARCIS'
run_root = work / 'diffstega_external_work'
subprocess.run(['bash', str(repo/'tools/bootstrap_diffstega_free_gpu.sh'), str(run_root)], cwd=repo, check=True)

In [ ]:
import pathlib, tarfile, hashlib, json
work = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path('/content')
run_root = work / 'diffstega_external_work'
bundle = work / 'NARCIS_TOMM_DiffStega_evidence.tar.gz'
with tarfile.open(bundle, 'w:gz') as tar:
    for rel in ['evidence', 'DiffStega/output']:
        p = run_root / rel
        if p.exists(): tar.add(p, arcname=rel)
h = hashlib.sha256(bundle.read_bytes()).hexdigest()
print(json.dumps({'bundle': str(bundle), 'bytes': bundle.stat().st_size, 'sha256': h}, indent=2))